In [ ]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')

**1. Install useful libraries and U-Net definition**


In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
from pathlib import Path
import cv2
import random
import math
from skimage.color import rgb2hed, hed2rgb

In [ ]:
# Show versioning of deep learning libraries
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Install albumentations
!pip install -q albumentations==1.3.1

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
!pip install torchstain

In [ ]:
# Define paths for K-Fold Cross Validation
current_dir = ""
k_fold_dir = os.path.join(current_dir, "k10_seed42_REINHARD")
baseline_dir = os.path.join(current_dir)
checkpoint_dir = os.path.join(baseline_dir, "checkpoint_REINHARD_DA_final")

# fold_00 è riservato per il test, quindi usiamo fold_01 a fold_09
available_folds = [f"fold_{i:02d}" for i in range(1, 10)]

print(f"K-Fold directory: {k_fold_dir}")
print(f"Available folds for training/validation: {available_folds}")
print(f"fold_00 riservato per il test")


In [ ]:
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(k_fold_dir, exist_ok=True)

In [ ]:
def find_best_target_image_global(k_fold_dir, train_folds, extension=".png"):
    """
    Cerca l'immagine target ideale analizzando tutte le immagini
    presenti nelle fold specificate per il training.
    """
    print(f"Analisi colori su tutte le fold di training: {train_folds}...")

    means_lab = []
    valid_files = []

    for fold in train_folds:
        image_dir = os.path.join(k_fold_dir, fold, "images")
        if not os.path.exists(image_dir):
            continue

        files = sorted([f for f in os.listdir(image_dir) if f.endswith(extension)])

        for f in files:
            path = os.path.join(image_dir, f)
            img = cv2.imread(path)
            if img is None: continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)

            # Calcolo media canali LAB
            means = [np.mean(img_lab[:,:,0]), np.mean(img_lab[:,:,1]), np.mean(img_lab[:,:,2])]
            means_lab.append(means)
            valid_files.append(path)

    if not valid_files:
        raise ValueError("Nessuna immagine trovata nelle fold di training specificate.")

    means_lab = np.array(means_lab)
    # Calcolo del centroide medio di tutto il dataset di training
    dataset_centroid = np.mean(means_lab, axis=0)

    # Trova l'immagine con la distanza euclidea minima dal centroide
    distances = np.linalg.norm(means_lab - dataset_centroid, axis=1)
    best_idx = np.argmin(distances)
    best_image_path = valid_files[best_idx]

    print(f"\n✅ Target Globale Trovato: {os.path.basename(best_image_path)}")
    print(f"📍 Percorso: {best_image_path}")

    return best_image_path

In [ ]:
class ReinhardNormalizer:
    def __init__(self, white_threshold=215):
        self.target_means = None
        self.target_stds = None
        self.fitted = False
        self.white_threshold = white_threshold

    def fit(self, target_img_rgb):
        # Conversione in LAB per calcolare le statistiche target
        img_lab = cv2.cvtColor(target_img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
        l_channel = img_lab[:, :, 0]
        mask_tissue = l_channel < self.white_threshold

        self.target_means = []
        self.target_stds = []
        for i in range(3):
            chan = img_lab[:, :, i]
            pixels = chan[mask_tissue]
            if len(pixels) == 0:
                self.target_means.append(np.mean(chan))
                self.target_stds.append(np.std(chan))
            else:
                self.target_means.append(np.mean(pixels))
                self.target_stds.append(np.std(pixels))
        self.fitted = True

    def normalize(self, image_rgb):
        img_lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
        l_src = img_lab[:, :, 0]
        background_mask = l_src > self.white_threshold

        res_channels = []
        for i in range(3):
            src_chan = img_lab[:, :, i]
            mask_tissue = ~background_mask
            if np.sum(mask_tissue) > 100:
                s_mean = np.mean(src_chan[mask_tissue])
                s_std = np.std(src_chan[mask_tissue])
            else:
                s_mean, s_std = np.mean(src_chan), np.std(src_chan)
            if s_std < 1e-5: s_std = 1
            norm = (src_chan - s_mean) * (self.target_stds[i] / s_std) + self.target_means[i]
            res_channels.append(norm)

        merged = cv2.merge(res_channels)
        merged = np.clip(merged, 0, 255).astype(np.uint8)
        out_rgb = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
        out_rgb[background_mask] = [255, 255, 255] # Forza bianco sullo sfondo
        return out_rgb

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# --- 1. CONFIGURAZIONE PERCORSI ---
base_dir = "/content/drive/MyDrive/Colab Notebooks/gabimachi"
input_kfold_dir = os.path.join(base_dir, "k10_seed42")         # Sorgente (Immagini Originali)
output_kfold_dir = os.path.join(base_dir, "k10_seed42_REINHARD") # Destinazione

# Definiamo le fold per l'analisi (da 01 a 09)
val_fold = "fold_05"
available_folds = [f"fold_{i:02d}" for i in range(1, 10)]
train_folds = [f for f in available_folds if f != val_fold]

# --- 2. RICERCA E CARICAMENTO TARGET ---
target_path = find_best_target_image_global(input_kfold_dir, train_folds)

# Carichiamo l'immagine fisica dal percorso trovato
target_img_bgr = cv2.imread(target_path)
if target_img_bgr is None:
    raise ValueError(f"Impossibile caricare l'immagine target: {target_path}")

# Convertiamo in RGB perché il tuo ReinhardNormalizer.fit si aspetta RGB
target_img_rgb = cv2.cvtColor(target_img_bgr, cv2.COLOR_BGR2RGB)

# --- 3. INIZIALIZZAZIONE REINHARD ---
normalizer = ReinhardNormalizer(white_threshold=215)
# ORA passiamo l'immagine (array), non il percorso (stringa)
normalizer.fit(target_img_rgb)

# --- 4. VISUALIZZAZIONE ---
plt.figure(figsize=(8, 8))
plt.imshow(target_img_rgb)
plt.title(f"Immagine Target (Riferimento Reinhard):\n{os.path.basename(target_path)}")
plt.axis('off')
plt.show()

print(f"Normalizzatore Reinhard addestrato con successo.")

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2
import torch # Non strettamente necessario qui, ma per sicurezza se richiamato altrove
import numpy as np
import os

def visualizza_esempi_reinhard(input_dir, train_folds, normalizer, n_esempi=5):
    # 1. Raccogli tutti i percorsi delle immagini dai fold di training (SORGENTE)
    all_image_paths = []
    for fold in train_folds:
        img_dir = os.path.join(input_dir, fold, "images")
        if os.path.exists(img_dir):
            files = [os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.endswith('.png')]
            all_image_paths.extend(files)

    if not all_image_paths:
        print("Errore: nessuna immagine trovata per la visualizzazione.")
        return

    # 2. Seleziona n_esempi casuali
    sample_paths = random.sample(all_image_paths, min(n_esempi, len(all_image_paths)))

    # 3. Creazione del plot
    fig, axes = plt.subplots(len(sample_paths), 2, figsize=(12, 5 * len(sample_paths)))

    # Se c'è solo un esempio, axes non è una matrice 2D, lo forziamo
    if len(sample_paths) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, path in enumerate(sample_paths):
        # Caricamento immagine originale (OpenCV legge BGR)
        img_bgr = cv2.imread(path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        try:
            # --- LOGICA REINHARD (CORRETTA) ---
            # Il nostro normalizer accetta RGB (NumPy) e restituisce RGB (NumPy)
            norm_img_np = normalizer.normalize(img_rgb)
            status = "Successo"
        except Exception as e:
            norm_img_np = img_rgb # In caso di errore mostra l'originale
            status = f"Errore: {str(e)}"

        # Visualizzazione Originale
        axes[i, 0].imshow(img_rgb)
        axes[i, 0].set_title(f"Originale: {os.path.basename(path)}")
        axes[i, 0].axis('off')

        # Visualizzazione Normalizzata
        axes[i, 1].imshow(norm_img_np)
        axes[i, 1].set_title(f"Reinhard Normalizzata ({status})")
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

# --- ESECUZIONE ---
# Nota: passiamo input_kfold_dir (dove sono le originali)
# Passa 'train_folds' (la lista), non il percorso di output!
visualizza_esempi_reinhard(input_kfold_dir, train_folds, normalizer, n_esempi=5)

**U-NET model**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """Two consecutive conv-batchnorm-relu layers"""
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_filters=64, depth=4, bilinear=True):
        super(UNet, self).__init__()
        self.depth = depth
        self.down_layers = nn.ModuleList()
        self.up_layers = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)

        # Encoder
        filters = init_filters
        for d in range(depth):
            conv = DoubleConv(in_channels, filters)
            self.down_layers.append(conv)
            in_channels = filters
            filters *= 2

        # Bottleneck
        self.bottleneck = DoubleConv(in_channels, filters)

        # Decoder
        for d in range(depth):
            filters //= 2
            if bilinear:
                up = nn.Sequential(
                    nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                    nn.Conv2d(filters * 2, filters, kernel_size=1)
                )
            else:
                up = nn.ConvTranspose2d(filters * 2, filters, kernel_size=2, stride=2)

            conv = DoubleConv(filters * 2, filters)
            self.up_layers.append(nn.ModuleDict({'up': up, 'conv': conv}))

        # Final layer
        self.final_conv = nn.Conv2d(init_filters, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder with skip connections
        skips = []
        for down in self.down_layers:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder with skip connections
        skips = skips[::-1]  # Reverse order
        for i, up_layer in enumerate(self.up_layers):
            x = up_layer['up'](x)
            skip = skips[i]

            # Handle size mismatch
            if x.shape != skip.shape:
                diff_h = skip.shape[2] - x.shape[2]
                diff_w = skip.shape[3] - x.shape[3]
                x = F.pad(x, [diff_w // 2, diff_w - diff_w // 2,
                             diff_h // 2, diff_h - diff_h // 2])

            x = torch.cat([skip, x], dim=1)
            x = up_layer['conv'](x)

        return self.final_conv(x)

**Loss Function - DiceBCELoss**

In [ ]:
class DiceBCELoss(nn.Module):
    """Combined Dice + BCE Loss from baseline100"""
    def __init__(self, weight=None, size_average=True):
        super(DiceBCELoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        # BCE component
        bce = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='mean')

        # Dice component
        inputs_prob = torch.sigmoid(inputs)
        inputs_prob = inputs_prob.view(-1)
        targets = targets.view(-1)
        intersection = (inputs_prob * targets).sum()
        dice_loss = 1 - (2.*intersection + smooth)/(inputs_prob.sum() + targets.sum() + smooth)

        return bce + dice_loss

**DSC Calculation Functions**

In [ ]:
def calculate_dice_score_single(pred_bin, target):
    """Calcola DSC per una SINGOLA immagine (già binarizzata)."""
    pred_flat = pred_bin.view(-1)
    target_flat = target.view(-1)

    intersection = (pred_flat * target_flat).sum().item()
    pred_sum = pred_flat.sum().item()
    target_sum = target_flat.sum().item()
    union = pred_sum + target_sum

    # DSC Standard (con smoothing)
    smooth = 1e-5
    if union == 0:
        dsc_standard = 1.0
    else:
        dsc_standard = (2. * intersection + smooth) / (union + smooth)

    # DSC Strict (senza smoothing)
    # Se il target è vuoto (maschera nera), restituiamo NaN per ignorarla nella media
    if target_sum == 0:
        dsc_strict = float('nan')
    else:
        dsc_strict = (2. * intersection) / union

    return {'dsc_standard': dsc_standard, 'dsc_strict': dsc_strict}

def calculate_dice_score(pred, target, threshold=0.5):
    """Calcola DSC per un intero batch."""
    if not isinstance(pred, torch.Tensor):
        pred = torch.from_numpy(pred)
    if not isinstance(target, torch.Tensor):
        target = torch.from_numpy(target)

    if pred.min() < 0 or pred.max() > 1:
        pred = torch.sigmoid(pred)

    pred_bin = (pred > threshold).float()

    if pred_bin.dim() == 3:
        pred_bin = pred_bin.unsqueeze(0)
        target = target.unsqueeze(0)

    batch_size = pred_bin.size(0)
    dsc_standard_list = []
    dsc_strict_list = []

    for i in range(batch_size):
        scores = calculate_dice_score_single(pred_bin[i], target[i])
        dsc_standard_list.append(scores['dsc_standard'])
        dsc_strict_list.append(scores['dsc_strict'])

    return {
        'dsc_standard_list': dsc_standard_list,
        'dsc_strict_list': dsc_strict_list
    }

**2. Dataset and Data Loader**

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGE_SIZE = 416

# Training transforms con le DA richieste
train_transform = A.Compose([
    # 1. Ridimensionamento (Resize)
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),

    # 2. Augmentation Geometriche (Vertical Flip + Random Rotate 90)
    # Aggiungiamo anche HorizontalFlip per completezza, come nel notebook HED
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),

    # 3. Flessione Elastica (Elastic Transform)
    # Parametri: alpha (intensità), sigma (smoothing), alpha_affine (trasformazione affine)
    A.ElasticTransform(
        alpha=1,
        sigma=50,
        alpha_affine=50,
        p=0.2                 # Probabilità di applicazione
    ),

    # 4. Normalizzazione (Normalization)
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5), max_pixel_value=255.0),

    # 5. Conversione in Tensore
    ToTensorV2(),
])

# Validation transforms (solo Resize e Normalize per coerenza)
val_transform = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5), max_pixel_value=255.0),
    ToTensorV2(),
])

In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader, ConcatDataset

class LiverDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        # Carica l'immagine (che è già normalizzata Macenko su disco)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.float32)

        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image_tensor = augmented["image"]
            mask_tensor = augmented["mask"]
            if mask_tensor.dim() == 2:
                mask_tensor = mask_tensor.unsqueeze(0)
        else:
            image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mask_tensor = torch.from_numpy(mask).float().unsqueeze(0)

        return image_tensor, mask_tensor, img_name

def create_fold_datasets(k_fold_dir, train_folds, val_fold, train_trans, val_trans):
    def get_paths(fold_name):
        f_path = os.path.join(k_fold_dir, fold_name)
        return os.path.join(f_path, "images"), os.path.join(f_path, "manual_py")

    train_datasets_list = [LiverDataset(*get_paths(f), transform=train_trans) for f in train_folds if os.path.exists(get_paths(f)[0])]
    val_dataset = LiverDataset(*get_paths(val_fold), transform=val_trans)

    return ConcatDataset(train_datasets_list), val_dataset

In [ ]:
# Dataset creation now handled in training loop for K-Fold rotation
batch_size = 8
print(f"Batch size: {batch_size}")


**3. Training Configuration**

In [ ]:
# Model parameters
init_filters = 64
n_epochs = 50
learning_rate = 1e-4

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize model
model = UNet(in_channels=3, out_channels=1, init_filters=init_filters, depth=4, bilinear=True)
model = model.to(device)

# Loss and optimizer
criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

print(f"\nModel Configuration:")
print(f"  Input channels: 3 (RGB - no preprocessing)")
print(f"  Initial filters: {init_filters}")
print(f"  Batch size: {batch_size}")
print(f"  Epochs: {n_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Loss function: DiceBCELoss")

**4. Save Network Parameters**

In [ ]:
# Save network configuration
config = {
    "model": "UNet",
    "init_filters": init_filters,
    "depth": 4,
    "bilinear": True,
    "batch_size": batch_size,
    "n_epochs": n_epochs,
    "learning_rate": learning_rate,
    "loss_function": "DiceBCELoss",
    "optimizer": "Adam",
    "in_channels": 3,
    "preprocessing": "none - RGB original images"
}

config_path = os.path.join(checkpoint_dir, 'network_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=4)

print(f"Configuration saved to: {config_path}")

**5. Training Loop**

In [ ]:
# --- CONFIGURAZIONE FISSA DEI FOLD ---
val_fold = "fold_05"
available_folds = [f"fold_{i:02d}" for i in range(1, 10)]
train_folds = [f for f in available_folds if f != val_fold]

os.makedirs(checkpoint_dir, exist_ok=True)

# Parametri e stato
threshold = 0.5
best_val_dice_standard = 0.0
early_stopping_patience = 30
patience_counter = 0
start_epoch = 0

print("="*60)
print(f"START TRAINING (Macenko Preprocessed)")
print(f"Validation: {val_fold} | Training: {train_folds}")
print(f"Salvataggio in: {checkpoint_dir}")
print("="*60)

for epoch in range(start_epoch, n_epochs):
    print(f"\nEPOCH {epoch+1}/{n_epochs}")

    train_dataset, val_dataset = create_fold_datasets(
        k_fold_dir, train_folds, val_fold, train_transform, val_transform
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # --- PHASE: TRAINING ---
    model.train()
    train_loss = 0.0
    for images, masks, names in tqdm(train_loader, desc="Training"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks) #
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)

    # --- PHASE: VALIDATION ---
    model.eval()
    val_dice_standard_list, val_dice_strict_list = [], []
    all_probs, all_names = [], []

    with torch.no_grad():
        for images, masks, names in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)

            res = calculate_dice_score(outputs, masks, threshold=threshold) #
            val_dice_standard_list.extend(res['dsc_standard_list'])
            val_dice_strict_list.extend(res['dsc_strict_list'])
            all_probs.append(torch.sigmoid(outputs).cpu().numpy())
            all_names.extend(names)

    # Calcolo metriche
    train_loss /= len(train_loader.dataset)
    dsc_standard = np.mean(val_dice_standard_list)
    dsc_strict = np.nanmean(val_dice_strict_list) if not np.all(np.isnan(val_dice_strict_list)) else 0.0

    print(f"  Results: Loss: {train_loss:.4f} | DSC Std: {dsc_standard:.4f} | DSC Strict: {dsc_strict:.4f}")

    # Logica di salvataggio record su DSC Standard
    if dsc_standard > best_val_dice_standard:
        print(f" NUOVO RECORD! {best_val_dice_standard:.4f} -> {dsc_standard:.4f}")
        best_val_dice_standard = dsc_standard
        patience_counter = 0

        # Salvataggio modello
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'dsc_standard': dsc_standard,
            'val_fold': val_fold
        }, os.path.join(checkpoint_dir, 'best_model.pth')) #

        # Salvataggio maschere predette per analisi visiva
        results_dir = os.path.join(checkpoint_dir, 'best_validation_masks')
        os.makedirs(results_dir, exist_ok=True)
        y_pred_all = (np.concatenate(all_probs) > threshold).astype(np.uint8)
        for i in range(len(all_names)):
            cv2.imwrite(os.path.join(results_dir, f"{os.path.splitext(all_names[i])[0]}_pred.png"), y_pred_all[i].squeeze() * 255)
    else:
        patience_counter += 1
        print(f"  Nessun miglioramento. Patience: {patience_counter}/{early_stopping_patience}")

    if patience_counter >= early_stopping_patience:
        print("\nEarly stopping raggiunto.")
        break